# Домашнее задание 9

In [1]:
import numpy as np


In [2]:
def prize_for(number):
    text = f"{number:03d}"
    payouts = [0]

    if text == "777":
        payouts.append(200)
    if text == "999":
        payouts.append(100)
    if text == "555":
        payouts.append(50)
    if text == "333":
        payouts.append(15)
    if text == "111":
        payouts.append(10)
    if text[1:] == "77":
        payouts.append(5)
    if text[2] == "7":
        payouts.append(3)
    if text[1:] == "00":
        payouts.append(2)
    if text[2] == "0":
        payouts.append(1)

    return max(payouts)


def prize_table():
    return np.array([prize_for(number) for number in range(1000)], dtype=float)


def exact_expected_payout():
    return prize_table().mean()


def simulate_payouts(games, seed, table):
    generator = np.random.default_rng(seed)
    numbers = generator.integers(0, 1000, size=games)
    return table[numbers]


def describe_result(payouts, bet=1.0):
    mean_payout = payouts.mean()
    standard_error = payouts.std(ddof=1) / np.sqrt(len(payouts))
    return {
        "игр": len(payouts),
        "средняя выплата": mean_payout,
        "средний чистый результат": mean_payout - bet,
        "стандартная ошибка": standard_error,
        "доверительный интервал": (mean_payout - 1.96 * standard_error,
                                   mean_payout + 1.96 * standard_error),
        "доля выигрышных игр, %": 100 * (payouts > 0).mean(),
    }


## Задание 1. Моделирование игры

In [3]:
games = 1_000_000
seed = 2026

table = prize_table()
payouts = simulate_payouts(games, seed, table)
result = describe_result(payouts)

print(f"Игр: {result['игр']:,}".replace(",", " "))
print(f"Средняя выплата: {result['средняя выплата']:.4f} руб.")
print(f"Средний чистый результат за игру: {result['средний чистый результат']:+.4f} руб.")
print(f"Стандартная ошибка: {result['стандартная ошибка']:.4f}")
print(f"95%-й интервал для средней выплаты: "
      f"[{result['доверительный интервал'][0]:.4f}; {result['доверительный интервал'][1]:.4f}]")
print(f"Доля выигрышных игр: {result['доля выигрышных игр, %']:.2f} %")

if result["средний чистый результат"] < 0:
    print("\nИгра игроку невыгодна: средний результат отрицательный.")
else:
    print("\nИгра игроку выгодна: средний результат положительный.")


Игр: 1 000 000
Средняя выплата: 0.7955 руб.
Средний чистый результат за игру: -0.2045 руб.
Стандартная ошибка: 0.0072
95%-й интервал для средней выплаты: [0.7813; 0.8096]
Доля выигрышных игр: 20.41 %

Игра игроку невыгодна: средний результат отрицательный.


## Задание 2. Теоретический расчёт

In [4]:
table = prize_table()
expected = exact_expected_payout()
values, counts = np.unique(table, return_counts=True)

print("=== Распределение выплат ===")
for payout, count in zip(values, counts):
    print(f"  {int(payout):4d} руб.: {count:4d} номеров, вероятность {count / 1000:.3f}")
print(f"\nСумма выплат по всем 1000 номерам: {table.sum():.0f} руб.")
print(f"Точное математическое ожидание выплаты: {expected:.4f} руб.")
print(f"Точный средний чистый результат за игру: {expected - 1:+.4f} руб.")

wins = int((table > 0).sum())
print(f"Вероятность выигрыша: {wins}/1000 = {wins / 1000:.3f}")


=== Распределение выплат ===
     0 руб.:  796 номеров, вероятность 0.796
     1 руб.:   90 номеров, вероятность 0.090
     2 руб.:   10 номеров, вероятность 0.010
     3 руб.:   90 номеров, вероятность 0.090
     5 руб.:    9 номеров, вероятность 0.009
    10 руб.:    1 номеров, вероятность 0.001
    15 руб.:    1 номеров, вероятность 0.001
    50 руб.:    1 номеров, вероятность 0.001
   100 руб.:    1 номеров, вероятность 0.001
   200 руб.:    1 номеров, вероятность 0.001

Сумма выплат по всем 1000 номерам: 800 руб.
Точное математическое ожидание выплаты: 0.8000 руб.
Точный средний чистый результат за игру: -0.2000 руб.
Вероятность выигрыша: 204/1000 = 0.204


## Сравнение

In [5]:
theory = exact_expected_payout()
simulation = describe_result(simulate_payouts(1_000_000, 2026, prize_table()))

difference = simulation["средняя выплата"] - theory
error = simulation["стандартная ошибка"]
z_score = difference / error

print(f"Точное ожидание выплаты:        {theory:.4f} руб.")
print(f"Моделирование:                  {simulation['средняя выплата']:.4f} руб.")
print(f"Разница:                        {difference:+.4f} руб.")
print(f"Стандартная ошибка:             {error:.4f}")
print(f"Разница в стандартных ошибках:  {z_score:+.2f}")
print(f"\nТеория: игрок теряет {1 - theory:.4f} руб. за игру.")


Точное ожидание выплаты:        0.8000 руб.
Моделирование:                  0.7955 руб.
Разница:                        -0.0045 руб.
Стандартная ошибка:             0.0072
Разница в стандартных ошибках:  -0.63

Теория: игрок теряет 0.2000 руб. за игру.
